# Inference Notebook

This notebook converts `models/inference.py` into runnable cells. It loads the local finetuned model and runs a small demo translation.

Adjust `model_dir` in the demo cell if your model is in a different path.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import List, Optional

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def load_model_and_tokenizer(model_dir: str, use_fp16: bool):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # choose dtype when loading to save memory on GPU
    load_kwargs = {}
    if device.type == "cuda" and use_fp16:
        load_kwargs["torch_dtype"] = torch.float16

    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir, **load_kwargs)
    model.to(device)
    model.eval()
    return model, tokenizer, device

def translate_texts(
    texts: List[str],
    model,
    tokenizer,
    device,
    src_lang: Optional[str],
    tgt_lang: Optional[str],
    max_length: int = 256,
    num_beams: int = 5,
):
    # Prepare tokenizer options
    # Many multilingual models require specifying src/tgt language in special ways.
    # If tokenizer exposes a language->id mapping, map target language to forced_bos_token_id.
    forced_bos_token_id = None
    if tgt_lang:
        # Try multiple methods to get the target language token ID
        if hasattr(tokenizer, "lang_code_to_id") and tgt_lang in tokenizer.lang_code_to_id:
            forced_bos_token_id = tokenizer.lang_code_to_id[tgt_lang]
        elif hasattr(tokenizer, "get_lang_id"):
            try:
                # some tokenizers implement get_lang_id(lang)
                forced_bos_token_id = tokenizer.get_lang_id(tgt_lang)
            except Exception:
                pass
        # Fallback: check if tgt_lang is directly in tokenizer vocabulary
        if forced_bos_token_id is None:
            try:
                forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
                # If it returns the unknown token ID, it's not valid
                if forced_bos_token_id == tokenizer.unk_token_id:
                    forced_bos_token_id = None
            except Exception:
                pass

    # Some tokenizers/models support specifying src language for tokenization (e.g., M2M100, NLLB)
    # CRITICAL: For NLLB/M2M100, must set src_lang BEFORE calling tokenizer
    if src_lang:
        if hasattr(tokenizer, "src_lang"):
            tokenizer.src_lang = src_lang
        if hasattr(tokenizer, "set_src_lang_special_tokens"):
            tokenizer.set_src_lang_special_tokens(src_lang)

    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)

    gen_kwargs = dict(max_length=max_length, num_beams=num_beams)
    if forced_bos_token_id is not None:
        gen_kwargs["forced_bos_token_id"] = forced_bos_token_id

    with torch.no_grad():
        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, **gen_kwargs)

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return decoded

/home/deeraj/downloaads1/nllbfinetune/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Demo run: adjust `model_dir` if needed
model_dir = "/home/deeraj/downloaads1/nllbfinetune/models/nllb-finetuned-de-en"
print("Model directory:", model_dir)

try:
    model, tokenizer, device = load_model_and_tokenizer(model_dir, use_fp16=False)
    texts = ["Die Verarbeitung großer Datenmengen in verteilten Systemen erfordert."]
    translated = translate_texts(texts, model, tokenizer, device, src_lang="deu_Latn", tgt_lang="_Latn", max_length=128, num_beams=4)
    for src, tgt in zip(texts, translated):
        print(f"> {src}\n=> {tgt}\n")
except Exception as e:
    import traceback
    print("Error during demo run:", e)
    traceback.print_exc()


Model directory: /home/deeraj/downloaads1/nllbfinetune/models/nllb-finetuned-de-en
> Die Verarbeitung großer Datenmengen in verteilten Systemen erfordert.
=> The processing of large amounts of data in distributed systems is required.

